In [ ]:
using LinearAlgebra, Printf
DEV = true
if DEV
    include("../src/EDKit.jl")
    using .EDKit
else
    using EDKit
end
using ITensors, ITensorMPS


# DAOE example

This notebook shows how to build the DAOE truncation MPO with `daoe(ps, l, gamma)` and apply it to Pauli-space MPS operators. The first half checks a few simple Pauli strings. The second half inserts the truncation into a small Heisenberg operator-growth calculation evolved with TEBD gates generated from `commutation_mat`.


In [ ]:
pad_labels(labels, L) = vcat(labels, fill("I", L - length(labels)))

function pauli_string_mps(ps, labels)
    productMPS(ps, pad_labels(labels, length(ps)))
end

function normalized_weight(D, psi)
    numer = real(inner(psi, apply(D, psi)))
    denom = real(inner(psi, psi))
    numer / denom
end


## Simple string weights

For `l = 2`, strings that spread beyond two active sites are suppressed by extra powers of `exp(-gamma)`. It is easiest to see the package's exact convention by evaluating a few strings directly.


In [ ]:
L = 6
ps = siteinds("Pauli", L)
gamma = 0.3
lcut = 2
D = daoe(ps, lcut, gamma)

samples = [
    ["I"],
    ["X"],
    ["X", "X"],
    ["X", "X", "X"],
    ["X", "Z"],
    ["X", "Z", "X"],
]

[(;
    string = join(labels),
    weight = normalized_weight(D, pauli_string_mps(ps, labels)),
) for labels in samples]


## Operator growth with TEBD

Here the operator is stored directly as a Pauli-space MPS. We evolve it with the Heisenberg generator `-i[H, .]`, written in Pauli coordinates by `commutation_mat`. The DAOE expectation value `<O|D|O> / <O|O>` then tracks how strongly the truncation would penalize the current operator.


In [ ]:
L = 8
ps = siteinds("Pauli", L)
dt = 0.05
steps = 24
gamma = 0.4
D2 = daoe(ps, 2, gamma)
D3 = daoe(ps, 3, gamma)

h2 = spin((1.0, "xx"), (1.0, "yy"), (1.0, "zz"))
superop = commutation_mat(h2)
gates = tebd4(fill(superop, L - 1), ps, dt)

psi = productMPS(ps, ["X", fill("I", L - 1)...])
times = collect(0:dt:steps * dt)
w2 = zeros(length(times))
w3 = zeros(length(times))

w2[1] = normalized_weight(D2, psi)
w3[1] = normalized_weight(D3, psi)
for n in 1:steps
    psi = apply(gates, psi)
    normalize!(psi)
    w2[n + 1] = normalized_weight(D2, psi)
    w3[n + 1] = normalized_weight(D3, psi)
end

(;
    times = times,
    l2_weight = w2,
    l3_weight = w3,
)


## Inserting the truncation into a time step

A minimal truncating loop is just "evolve, apply `D`, renormalize". The exact cutoff schedule is model-dependent, but the mechanics are this simple.


In [ ]:
bond_dims(psi) = [linkdim(psi, b) for b in 1:length(psi)-1]

psi_raw = productMPS(ps, ["X", fill("I", L - 1)...])
psi_trunc = copy(psi_raw)

for _ in 1:steps
    psi_raw = apply(gates, psi_raw)
    normalize!(psi_raw)

    psi_trunc = apply(gates, psi_trunc)
    psi_trunc = apply(D2, psi_trunc)
    normalize!(psi_trunc)
end

(;
    raw_weight = normalized_weight(D2, psi_raw),
    truncated_weight = normalized_weight(D2, psi_trunc),
    raw_max_bond = maximum(bond_dims(psi_raw)),
    truncated_max_bond = maximum(bond_dims(psi_trunc)),
)
